In [1]:
import torch
import torchvision
from torch import nn

/home/zhang402/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initialize DNN model and compile using TorchInductor

In [2]:
# model = torchvision.models.resnet50(pretrained=True).to("cuda")
device = (
   "cuda"
   if torch.cuda.is_available()
   else "mps"
   if torch.backends.mps.is_available()
   else "cpu"
)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [3]:
torch._dynamo.reset()
resnet50_compiled = torch.compile(
        model,
        options={
            "trace.enabled": True,
        },
)

Set up the training loop

In [4]:
# for this example, we generate one random sample
inputs = torch.randn(64, 784).to("cuda")
labels = torch.randn(64, 10).to("cuda")

# initialize the loss calculation and optimizer
learning_rate = 0.001
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet50_compiled.parameters(), lr=learning_rate)

Wrap optimizer.step() in torch.compile()

In [5]:
def optimizer_step_fn(optimizer):
    '''Return torch.compile'd version of optimizer.step()'''
    def f():
        optimizer.step()
    return torch.compile(
        f,
        options={
            "trace.enabled": True,
        },
    )

optimizer_step = optimizer_step_fn(optimizer)

Run one training iteration

In [6]:
# Zero out the optimizer
optimizer.zero_grad()

# Forward pass
outputs = resnet50_compiled(inputs)
loss = criterion(outputs, labels) # torch.nn.CrossEntropyLoss()
# outputs, loss = forward(inputs, labels)

# Backward pass
loss.backward()

# parameter update
optimizer_step()

/home/zhang402/.local/lib/python3.8/site-packages/torch/_inductor/compile_fx.py:140: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
[2024-03-22 21:59:08,995] [0/0] torch._inductor.debug: [WARNING] model__0_forward_10 debug trace: /tmp/torchinductor_zhang402/sj/csjipcd2jksblpaqzsfs6bcv25ec7drl3nlxuaoslhmfamfjjevd.debug
[2024-03-22 21:59:10,153] torch._inductor.debug: [WARNING] model__0_backward_12 debug trace: /tmp/torchinductor_zhang402/ys/cys3wihllduuklvnok4axnpg57b2hfqzr3wez3s5ivw5gawphxjs.debug
[2024-03-22 21:59:12,906] [2/0] torch._inductor.debug: [WARNING] model__4_inference_13 debug trace: /tmp/torchinductor_zhang402/w6/cw6lqjdjrhot5gafmmzo3kqawbe42auhjhaflwkb4aodvfbllbjx.debug
